## Preparation (mapping isoforms back to genes)

In [2]:
import os
import csv
import pandas as pd
import re
import subprocess
import numpy as np
import scipy.stats as stats

new_dir = "/home/jingqi/RNALocateV3.0"
os.chdir(new_dir)
os.getcwd()

'/home/jingqi/RNALocateV3.0'

### Mapping

In [4]:
input_file = "Data/Raw/3UTR_clean.txt"
output_file = "Data/Raw/gene_3UTR_matrix.csv"

print(f"Reading from: {input_file}")

gene_to_transcripts = {}

with open(input_file, 'r') as f:
    for line in f:
        if line.startswith(">"):
            clean_line = line.strip()[1:] 
            parts = clean_line.split('|')
            
            if len(parts) >= 2:
                # Corrected index mapping
                gene_symbol = parts[0]
                transcript_id = parts[1]
                
                if gene_symbol not in gene_to_transcripts:
                    gene_to_transcripts[gene_symbol] = []
                    
                # Prevent identical transcript duplicates
                if transcript_id not in gene_to_transcripts[gene_symbol]:
                    gene_to_transcripts[gene_symbol].append(transcript_id)

max_transcripts = 0
rows = []

for gene, transcripts in gene_to_transcripts.items():
    if len(transcripts) > max_transcripts:
        max_transcripts = len(transcripts)
    
    row = [gene] + transcripts
    rows.append(row)

print(f"Found {len(rows)} unique genes.")
print(f"Max transcripts for one gene: {max_transcripts}")

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    
    header = ["Gene_Symbol"] + [f"Transcript_{i+1}" for i in range(max_transcripts)]
    writer.writerow(header)
    
    writer.writerows(rows)


Reading from: Data/Raw/3UTR_clean.txt
Found 713 unique genes.
Max transcripts for one gene: 18


### little validation (can skip)

In [32]:
final_gene_list = []
with open("Data/Raw/final_gene_list.txt", "r") as f:
    for line in f:
        gene = line.strip()
        if gene:
            final_gene_list.append(gene)

original_set = set(final_gene_list)

matrix_df = pd.read_csv("Data/Raw/gene_transcript_matrix.csv")
found_set = set(matrix_df['Gene_Symbol'].unique())

missing_genes = original_set - found_set
print(missing_genes)

736
728
7
{'ENSMUSG00000125456', 'ENSMUSG00000121480', 'ENSMUSG00000143377', 'ENSMUSG00000132887', 'ENSMUSG00000135935', '2810004N23Rik', 'ENSMUSG00000124710'}


## Assign SCL on to isoforms

### Extract the motifs validated

In [5]:
# the file "protein_tomtom_longer_filtered.csv" wouldn't work. weird and don't know why
# copied and renamed
df = pd.read_csv("Data/TOMTOM_New/RBP_filtered.csv")

# Get unique target motif IDs
targets = set(df['Target'].dropna().unique())

print(f"Unique targets: {len(targets)}")

Unique targets: 17


### Get the seqeunces from database

In [15]:
def extract_motifs_from_meme_db(db_file, target_ids, output_file):
    """
    Extract specific motifs from a MEME database file.
    """
    with open(db_file, 'r') as f:
        content = f.read()
    
    # Split by MOTIF
    parts = re.split(r'(?=^MOTIF )', content, flags=re.MULTILINE)
    
    # Keep header (first part before any MOTIF)
    header = parts[0] if not parts[0].startswith('MOTIF') else ""
    
    # Find motifs matching the target IDs
    kept_motifs = []
    for part in parts:
        if part.startswith('MOTIF'):
            # Extract motif ID (first word after MOTIF)
            match = re.match(r'MOTIF\s+(\S+)', part)
            if match:
                motif_id = match.group(1)
                if motif_id in target_ids:
                    kept_motifs.append(part)
    
    # Write subset file
    with open(output_file, 'w') as f:
        f.write(header)
        for motif in kept_motifs:
            f.write(motif)
    
    print(f"\n Saved {len(kept_motifs)} motifs to {output_file}")
    return len(kept_motifs)

# Path to original database (same one used for TOMTOM)
databases = {
    "CISBP_RNA": "~/meme_db/motif_databases/CISBP-RNA/Mus_musculus.dna_encoded.meme",
}

## Extract subset
# os.makedirs("Data/FIMO_New", exist_ok=True)

for db_name, db_path in databases.items():
    full_path = os.path.expanduser(db_path)
    output_path = f"Data/FIMO_New/filtered_{db_name}.meme"
    extract_motifs_from_meme_db(full_path, targets, output_path)




 Saved 17 motifs to Data/FIMO_New/filtered_CISBP_RNA.meme


### Run FIMO (all transcripts against identified motifs)

In [1]:
import subprocess
import re

input_fasta = "Data/Raw/3UTR_clean.txt"
safe_fasta = "Data/Raw/3UTR_safe.txt"
bfile_path = "Data/MEME_New/background_model.bfile"

with open(input_fasta, "r") as f_in, open(safe_fasta, "w") as f_out:
    for line in f_in:
        line = line.strip()
        if not line:
            continue
            
        if line.startswith(">"):
            header = line.split()[0]
            if len(header) > 50:
                header = header[:50]
            f_out.write(header + "\n")
        else:
            clean_seq = re.sub(r'[^A-Za-z]', '', line)
            if clean_seq:
                f_out.write(clean_seq + "\n")

fimo_path = "/home/jingqi/miniconda3/envs/rnalocate_old/bin/fimo"
output_dir = "Data/FIMO_New/RBP_all_transcripts"
meme_file = "Data/FIMO_New/filtered_CISBP_RNA.meme"

fimo_cmd = [
    fimo_path,
    "--oc", output_dir,
    "--thresh", "0.0001",
    "--norc",
    "--bgfile", bfile_path,
    meme_file,
    safe_fasta
]

print("Executing FIMO with sanitized headers and explicit background model...")
subprocess.run(fimo_cmd, check=True)

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Raw/3UTR_clean.txt'

In [22]:
# CORRECT HEADER
fimo_cols = [
    "motif_id", "sequence_name", "start", "stop", "strand", 
    "score", "p-value", "q-value", "matched_sequence"
]

fimo_df = pd.read_csv(
    "Data/FIMO_New/RBP_all_transcripts/fimo.txt", 
    sep='\t', 
    comment='#', 
    header=None, 
    names=fimo_cols
)

filtered_df = fimo_df[fimo_df['q-value'] < 0.05]

print(f"n\ Hits with q-value < 0.05: {len(filtered_df)}")

gene_lookup = {}

df = pd.read_csv("Data/Raw/gene_3UTR_matrix.csv")

# Iterate row by row to build the dictionary
for index, row in df.iterrows():
    gene_symbol = row['Gene_Symbol']
    transcripts = row.drop('Gene_Symbol').dropna()
    
    for transcript in transcripts:
        if isinstance(transcript, str):
            ids = transcript.strip()
            gene_lookup[ids] = gene_symbol

print(f"Loaded {len(gene_lookup)} gene-transcript mappings")
print(f"Unique Genes found: {df['Gene_Symbol'].nunique()}")

df_scl = pd.read_csv("Data/TOMTOM_New/RBP_filtered.csv")

# Combine and create mapping class, bucket
motif_to_cluster = {}


for _, row in df_scl.iterrows():
    target = row['Target']
    cluster = f"{row['Class']}_{row['Bucket']}"
    if target not in motif_to_cluster:
        motif_to_cluster[target] = cluster

print(f"Mapped {len(motif_to_cluster)} motifs to clusters")


prob_df = pd.read_csv("Data/Main/Probabilities.csv")
prob_df = prob_df.set_index('sequence_id')

prob_cols = {
    'chromatin': 'chromatin_prob',
    'cytoplasm': 'cytoplasm_prob',
    'cytosol': 'cytosol_prob',
    'ER': 'endoplasmic reticulum_prob',
    'extracellular': 'extracellular region_prob',
    'membrane': 'membrane_prob',
    'mitochondrion': 'mitochondrion_prob',
    'nucleolus': 'nucleolus_prob',
    'nucleoplasm': 'nucleoplasm_prob',
    'nucleus': 'nucleus_prob',
    'ribosome': 'ribosome_prob'
}

results = []

for _, row in filtered_df.iterrows():
    motif_id = row['motif_id']
    transcript = row['sequence_name']
    p_value = row['p-value']
    q_value = row['q-value']
    
    # Get gene for this transcript
    gene = gene_lookup.get(transcript, 'Unknown')
    
    # Get cluster for this motif
    cluster = motif_to_cluster.get(motif_id, None)
    if not cluster:
        continue
    
    # Parse class and bucket
    parts = cluster.rsplit('_', 1)
    if len(parts) != 2:
        continue
    class_name, bucket = parts
    
    # Get transcript probability
    if transcript not in prob_df.index:
        continue
    
    # Find the probability column
    prob_col = prob_cols.get(class_name, None)
    if not prob_col:
        for col in prob_df.columns:
            if class_name.lower() in col.lower():
                prob_col = col
                break
    
    if not prob_col:
        continue
    
    prob_value = prob_df.loc[transcript, prob_col]
    
    # Validate based on bucket
    if bucket == 'positive':
        expected = "High"
        match = prob_value > 0.5
    else:
        expected = "Low"
        match = prob_value < 0.5
    
    results.append({
        'Gene': gene,
        'Transcript': transcript,
        'Motif': motif_id,
        'Cluster': cluster,
        'Probability': prob_value,
        'Expected': expected,
        'Match': '✓' if match else '✗',
        'q_value': q_value
    })

result_df = pd.DataFrame(results)

# Sort: genes alphabetically, Unknown at the end
result_df['_sort_key'] = result_df['Gene'].apply(lambda x: (1, '') if x == 'Unknown' else (0, x.lower()))
result_df = result_df.sort_values(by=['_sort_key', 'Gene', 'Transcript'])
result_df = result_df.drop(columns=['_sort_key'])

# Reset index
result_df = result_df.reset_index(drop=True)

# Save
result_df.to_csv("Data/FIMO_New/motifs_validated.csv", index=False)
print(f"\n Total validated hits: {len(result_df)}")

# Keep only the validated matches (✓)
clean_df = result_df[result_df['Match'] == '✓'].copy()
clean_df = clean_df[result_df['Gene'] != 'Unknown']

# Drop the validation columns 
cols_to_drop = ['Probability', 'Expected', 'Match']
clean_df = clean_df.drop(columns=cols_to_drop, errors='ignore')

output_file = "Data/FIMO_New/motifs_validated_filtered.csv"
clean_df.to_csv(output_file, index=False)

if not result_df.empty:
    matches = (result_df['Match'] == '✓').sum()
    total = len(result_df)
    print(f"Matching predictions: {matches}/{total} ({100*matches/total:.1f}%)")
    
    # Gene stats
    print(f"Unique genes: {result_df[result_df['Gene'] != 'Unknown']['Gene'].nunique()}")
    
    print(f"\nBy cluster:")
    for cluster in sorted(result_df['Cluster'].unique()):
        subset = result_df[result_df['Cluster'] == cluster]
        m = (subset['Match'] == '✓').sum()
        t = len(subset)
        print(f"  {cluster}: {m}/{t} ({100*m/t:.1f}%)")


n\ Hits with q-value < 0.05: 66
Loaded 2431 gene-transcript mappings
Unique Genes found: 713
Mapped 17 motifs to clusters


KeyError: 'Gene'

In [82]:
# compress all the same motifs, check if one motif hits two or more localization

input_file = "Data/FIMO/motifs_validated_filtered.csv"
df = pd.read_csv(input_file)
# This converts "1.2e-5" strings to numbers
df['q_value'] = pd.to_numeric(df['q_value'])

#  Define Fisher's Method Function
def fisher_score(group):
    q_values = group['q_value'].values
    # Calculate Chi-Squared Statistic: -2 * sum(ln(q))
    chi2_stat = -2 * np.sum(np.log(q_values))
    # Degrees of freedom = 2 * number of tests
    df_chi2 = 2 * len(q_values)
    # Calculate combined p-value (Upper tail of Chi2 distribution)
    p_val = stats.chi2.sf(chi2_stat, df_chi2)
    return pd.Series({'Hits': len(q_values), 'Fisher_score': chi2_stat, 'p_adjusted': p_val})

# Filter out motifs that appear in more than one cluster
cluster_counts = df.groupby('Motif')['Cluster'].nunique()
valid_motifs = cluster_counts[cluster_counts == 1].index

# Filter the main dataframe to keep only these valid motifs
df_valid = df[df['Motif'].isin(valid_motifs)].copy()

print(f"  - Original unique motifs: {df['Motif'].nunique()}")
print(f"  - Motifs with conflicting clusters (discarded): {df['Motif'].nunique() - len(valid_motifs)}")
print(f"  - Motifs kept: {len(valid_motifs)}")

# Group by motif and cluster 
agg_df = df_valid.groupby(['Motif', 'Cluster']).apply(fisher_score).reset_index()

# Sort
agg_df = agg_df.sort_values('Fisher_score', ascending = False)

# Format the Output
agg_df = agg_df[['Motif', 'Cluster', 'Hits', 'Fisher_score', 'p_adjusted']]

output_file = "Data/FIMO/a_aggregated.csv"
agg_df.to_csv(output_file, index=False)

print(f"\n Aggregated file saved to: {output_file}")

  - Original unique motifs: 89
  - Motifs with conflicting clusters (discarded): 0
  - Motifs kept: 89

Success! Aggregated file saved to: Data/FIMO/motifs_aggregated.csv


In [91]:
# compress all the transcripts with the same motif, at least one hit being true is enough

df = pd.read_csv("Data/FIMO/motifs_validated_filtered.csv")

# Ensure numeric types and clean data
df['q_value'] = pd.to_numeric(df['q_value'])
df = df.dropna(subset=['q_value'])

def combined_pval(group):
    q_values = group['q_value'].values
    log_qs = np.log(q_values) 
    prob_all_false = np.exp(np.sum(log_qs))
    # Phred Score -logP
    phred_score = -(np.sum(log_qs) / np.log(10))
    return  pd.Series({
        'Counts': len(q_values),
        'p_adjusted': prob_all_false,
        'Phred_score': phred_score
    })

# We keep Site_Count and Best_Site_Q as useful secondary metrics
compressed_df = df.groupby(['Gene', 'Transcript', 'Motif', 'Cluster']).apply(combined_pval).reset_index()

# sorting, plus cluter the rows with the same gene together
compressed_df = compressed_df.sort_values(by=['Gene', 'Phred_score'], ascending=[True, False])
# compressed_df = compressed_df.sort_values('Phred_score', ascending = False)

output_file = "Data/FIMO/a_compressed_in_transcripts.csv"
compressed_df.to_csv(output_file, index=False)

print(f"Compressed {len(df)} hits into {len(compressed_df)} unique transcript-motif pairs.")
print(f"Results saved to: {output_file}")

Compressed 29720 hits into 10450 unique transcript-motif pairs.
Results saved to: Data/FIMO/a_compressed_in_transcripts.csv


In [31]:
fimo_file = "Data/FIMO_New/RBP_all_transcripts/fimo.txt"
output_filtered_csv = "Data/FIMO_New/filtered_fimo_results.csv"

fimo_df = pd.read_csv(fimo_file, sep='\t', skipfooter=4, engine='python')

fimo_df.columns = fimo_df.columns.str.replace('^#\\s*', '', regex=True).str.strip().str.replace(' ', '_')
fimo_df.rename(columns={'pattern_name': 'motif_id'}, inplace=True)

if 'q-value' in fimo_df.columns:
    filtered_fimo = fimo_df[fimo_df['q-value'] < 0.05]
elif 'p-value' in fimo_df.columns:
    filtered_fimo = fimo_df[fimo_df['p-value'] < 0.0001]
else:
    raise ValueError(f"Failed to locate statistical columns. Headers: {fimo_df.columns.tolist()}")

filtered_fimo.to_csv(output_filtered_csv, index=False)

print(f"Raw binding events: {len(fimo_df)}")
print(f"High-confidence events: {len(filtered_fimo)}")
print(f"File saved to: {output_filtered_csv}")

Raw binding events: 21399
High-confidence events: 66
File saved to: Data/FIMO_New/filtered_fimo_results.csv


### Merge and Filter

In [40]:
from scipy.stats import combine_pvalues

fimo_file = "Data/FIMO_New/RBP_all_transcripts/fimo.txt"
output_filtered_csv = "Data/FIMO_New/transcript_level_fimo.csv"
cisbp_db_path = os.path.expanduser("~/meme_db/motif_databases/CISBP-RNA/Mus_musculus.dna_encoded.meme")

id_to_name = {}
try:
    with open(cisbp_db_path, 'r') as f:
        for line in f:
            if line.startswith("MOTIF"):
                parts = line.strip().split()
                if len(parts) >= 3:
                    id_to_name[parts[1]] = parts[2]
except FileNotFoundError:
    pass

fimo_df = pd.read_csv(fimo_file, sep='\t', skipfooter=4, engine='python')
fimo_df.columns = fimo_df.columns.str.replace('^#\\s*', '', regex=True).str.strip().str.replace(' ', '_')
fimo_df.rename(columns={'pattern_name': 'motif_id'}, inplace=True)

def fisher_combine(p_array):
    p_vals = np.asarray(p_array)
    p_vals = np.where(p_vals == 0, 1e-300, p_vals)
    if len(p_vals) == 1:
        return float(p_vals[0])
    return combine_pvalues(p_vals, method='fisher')[1]

agg_df = fimo_df.groupby(['sequence_name', 'motif_id']).agg(
    site_count=('p-value', 'count'),
    transcript_p_value=('p-value', fisher_combine)
).reset_index()

raw_names = agg_df['motif_id'].map(id_to_name)
agg_df['Protein_Name'] = raw_names.str.extract(r'^\(?([a-zA-Z0-9]+)\)?', expand=False).str.capitalize()

cols = ['sequence_name', 'motif_id', 'RBP_Name', 'site_count', 'transcript_p_value']
agg_df = agg_df[cols]

transcript_p_thresh = 1e-6
final_df = agg_df[agg_df['transcript_p_value'] < transcript_p_thresh].copy()

final_df.to_csv(output_filtered_csv, index=False)

print(f"Unique Transcript-Motif pairs analyzed: {len(agg_df)}")
print(f"Highly significant transcript-level events (p < {transcript_p_thresh}): {len(final_df)}")
print(f"Data saved to {output_filtered_csv}")

Unique Transcript-Motif pairs analyzed: 9456
Highly significant transcript-level events (p < 1e-06): 3776
Data saved to Data/FIMO_New/transcript_level_fimo.csv


### Profiling(can ignore)

In [41]:
filtered_file = "Data/FIMO_New/transcript_level_fimo.csv"

# Load the filtered data
try:
    df = pd.read_csv(filtered_file)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find {filtered_file}. Ensure the previous filtering block completed successfully.")

total_transcripts = df['sequence_name'].nunique()
print(f"Total Significant Transcript-RBP Interactions: {len(df)}")
print(f"Total Unique Transcripts Bound: {total_transcripts}\n")

print("=== RBP TARGET DISTRIBUTION ===")
# Aggregate statistics per RBP
rbp_stats = df.groupby(['motif_id', 'Protein_Name']).agg(
    Bound_Transcripts=('sequence_name', 'nunique'),
    Total_Sites=('site_count', 'sum'),
    Mean_Sites_Per_Transcript=('site_count', 'mean')
).reset_index()

# Sort by the most widely acting RBPs
rbp_stats = rbp_stats.sort_values(by='Bound_Transcripts', ascending=False)

# Calculate penetrance (what % of all bound transcripts does this RBP target?)
rbp_stats['Coverage (%)'] = (rbp_stats['Bound_Transcripts'] / total_transcripts * 100).round(2).astype(str) + '%'

# Format the mean for readability
rbp_stats['Mean_Sites_Per_Transcript'] = rbp_stats['Mean_Sites_Per_Transcript'].round(2)

# Rearrange columns for a clean print
display_cols = ['Protein_Name', 'Bound_Transcripts', 'Coverage (%)', 'Total_Sites', 'Mean_Sites_Per_Transcript']
print(rbp_stats[display_cols].to_string(index=False))

print("\n=== COMBINATORIAL REGULATION (RBPs per Transcript) ===")
# How many unique RBPs map to each transcript?
complexity = df.groupby('sequence_name')['motif_id'].nunique()

print(complexity.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).to_string())

# Print a quick distribution of the complexity
print("\nTranscript Complexity Breakdown:")
complexity_counts = complexity.value_counts().sort_index()
for num_rbps, count in complexity_counts.items():
    print(f"Transcripts bound by exactly {num_rbps} unique RBP(s): {count}")

Total Significant Transcript-RBP Interactions: 3776
Total Unique Transcripts Bound: 1119

=== RBP TARGET DISTRIBUTION ===
Protein_Name  Bound_Transcripts Coverage (%)  Total_Sites  Mean_Sites_Per_Transcript
      Elavl2                497       44.41%         1914                       3.85
       Rbm38                349       31.19%         2689                       7.70
       Pcbp2                330       29.49%          999                       3.03
       Celf4                310        27.7%         2467                       7.96
       Celf3                310        27.7%         2467                       7.96
        Tia1                282        25.2%          769                       2.73
       Cpeb4                254        22.7%          656                       2.58
        Raly                220       19.66%          591                       2.69
       Pcbp1                211       18.86%          515                       2.44
     Hnrnph2                

### Map to SCL

#### Ensure the RBP difference

In [44]:
filtered_file = "Data/FIMO_New/transcript_level_fimo.csv"
output_matrix = "Data/FIMO_New/Isoform_Variance_Matrix.csv"

df = pd.read_csv(filtered_file)

# Split headers
split_names = df['sequence_name'].str.split('|', expand=True)
df['Gene'] = split_names[0]
df['Transcript'] = split_names[1]

# Pivot into horizontal matrix
matrix = df.pivot_table(
    index=['Gene', 'Transcript'], 
    columns='Protein_Name', 
    values='site_count', 
    fill_value=0
).reset_index()

matrix.columns.name = None
rbp_columns = [col for col in matrix.columns if col not in ['Gene', 'Transcript']]

# Convert all counts to binary (1 if > 0, else 0)
matrix[rbp_columns] = (matrix[rbp_columns] > 0).astype(int)

def has_binary_variance(group):
    # Drop genes with only 1 transcript
    if len(group) == 1:
        return False
        
    # Check if the binary presence/absence profiles differ between isoforms
    unique_profiles = group[rbp_columns].drop_duplicates()
    return len(unique_profiles) > 1

# Apply the variance filter
dynamic_genes_matrix = matrix.groupby('Gene').filter(has_binary_variance)

dynamic_genes_matrix.to_csv(output_matrix, index=False)

print(f"Initial transcripts analyzed: {len(matrix)}")
print(f"Transcripts retained after dropping uniform genes: {len(dynamic_genes_matrix)}")
print(f"Total unique genes demonstrating binary RBP variance: {dynamic_genes_matrix['Gene'].nunique()}")
print(f"Matrix saved to {output_matrix}")

Initial transcripts analyzed: 1119
Transcripts retained after dropping uniform genes: 440
Total unique genes demonstrating binary RBP variance: 127
Matrix saved to Data/FIMO_New/Isoform_Variance_Matrix.csv


#### Ensure the SCL difference

In [45]:
variance_matrix_file = "Data/FIMO_New/Isoform_Variance_Matrix.csv"
probabilities_file = "Data/Main/Probabilities.csv" 
output_file = "Data/FIMO_New/Functional_Zipcode_Candidates.csv"

# Load the data
matrix_df = pd.read_csv(variance_matrix_file)
prob_df = pd.read_csv(probabilities_file)

# Extract the RBP profile as a readable string
rbp_columns = [col for col in matrix_df.columns if col not in ['Gene', 'Transcript']]

def get_rbp_profile(row):
    # Keep RBPs where the binary value is 1
    bound_rbps = [rbp for rbp in rbp_columns if row[rbp] == 1]
    return ", ".join(bound_rbps) if bound_rbps else "None"

matrix_df['RBP_Profile'] = matrix_df.apply(get_rbp_profile, axis=1)

# Clean Transcript IDs for safe merging 
# (In case one file uses ENSMUST...1 and the other uses ENSMUST... without the version)
matrix_df['Clean_Transcript'] = matrix_df['Transcript'].str.split('.').str[0]
prob_df['Clean_Transcript'] = prob_df['sequence_id'].str.split('.').str[0]

# Merge the datasets
merged_df = pd.merge(matrix_df, prob_df, on='Clean_Transcript', how='inner')

# Extract the Subcellular Localization (SCL) profile using the strict > 0.75 threshold
prob_columns = [col for col in prob_df.columns if col.endswith('_prob')]

def get_scl_profile(row):
    locations = []
    for col in prob_columns:
        if row[col] > 0.75:
            # Strip the '_prob' suffix for a clean biological name
            loc_name = col.replace('_prob', '')
            locations.append(loc_name)
    return ", ".join(locations) if locations else "Unclassified"

merged_df['SCL_Profile'] = merged_df.apply(get_scl_profile, axis=1)

# Filter for SCL variance within the same gene
def has_scl_variance(group):
    # Drop genes where every transcript goes to the exact same compartment(s)
    unique_scls = group['SCL_Profile'].drop_duplicates()
    return len(unique_scls) > 1

final_candidates = merged_df.groupby('Gene').filter(has_scl_variance)

# Format the final output
final_output = final_candidates[['Gene', 'Transcript', 'RBP_Profile', 'SCL_Profile']].copy()

# Sort for readability
final_output.sort_values(by=['Gene', 'Transcript'], inplace=True)

# Export to CSV
final_output.to_csv(output_file, index=False)

print(f"Total transcripts inputted from variance matrix: {len(matrix_df)}")
print(f"Transcripts successfully mapped to prediction file: {len(merged_df)}")
print(f"Transcripts retained after dropping uniform SCL genes: {len(final_output)}")
print(f"Total high-confidence candidate genes: {final_output['Gene'].nunique()}")
print(f"\nFinal dataset saved to: {output_file}")

Total transcripts inputted from variance matrix: 440
Transcripts successfully mapped to prediction file: 440
Transcripts retained after dropping uniform SCL genes: 401
Total high-confidence candidate genes: 112

Final dataset saved to: Data/FIMO_New/Functional_Zipcode_Candidates.csv


#### Merge the overlapping regions as well as paralougs and we'll see

In [47]:
variance_matrix_file = "Data/FIMO_New/Isoform_Variance_Matrix.csv"
probabilities_file = "Data/Main/Probabilities.csv"
output_file = "Data/FIMO_New/Adjusted_Functional_Zipcode_Candidates.csv"

matrix_df = pd.read_csv(variance_matrix_file)
prob_df = pd.read_csv(probabilities_file)

# Extract RBP Profile
rbp_columns = [col for col in matrix_df.columns if col not in ['Gene', 'Transcript']]

def get_rbp_profile(row):
    bound_rbps = [rbp for rbp in rbp_columns if row[rbp] == 1]
    return ", ".join(bound_rbps) if bound_rbps else "None"

matrix_df['RBP_Profile'] = matrix_df.apply(get_rbp_profile, axis=1)

# Clean IDs and Merge
matrix_df['Clean_Transcript'] = matrix_df['Transcript'].str.split('.').str[0]
prob_df['Clean_Transcript'] = prob_df['sequence_id'].str.split('.').str[0]

merged_df = pd.merge(matrix_df, prob_df, on='Clean_Transcript', how='inner')

# Evaluate the Adjusted SCL Profile
def get_adjusted_scl_profile(row):
    locations = []
    
    # 1. Macro-Cluster: Nucleus
    nucleus_cols = ['chromatin_prob', 'nucleolus_prob', 'nucleoplasm_prob', 'nucleus_prob']
    if any(row.get(col, 0) > 0.75 for col in nucleus_cols):
        locations.append('Nucleus')
        
    # 2. Macro-Cluster: Cytoplasm
    cyto_cols = ['cytoplasm_prob', 'cytosol_prob']
    if any(row.get(col, 0) > 0.75 for col in cyto_cols):
        locations.append('Cytoplasm')
        
    # 3. Independent Compartments
    independent_cols = {
        'endoplasmic reticulum_prob': 'Endoplasmic reticulum',
        'extracellular region_prob': 'Extracellular region',
        'membrane_prob': 'Membrane',
        'mitochondrion_prob': 'Mitochondrion',
        'ribosome_prob': 'Ribosome'
    }
    
    for col, name in independent_cols.items():
        if row.get(col, 0) > 0.75:
            locations.append(name)
            
    return ", ".join(locations) if locations else "Unclassified"

merged_df['Adjusted_SCL_Profile'] = merged_df.apply(get_adjusted_scl_profile, axis=1)

# Filter for variance using the new adjusted profiles
def has_adjusted_scl_variance(group):
    unique_scls = group['Adjusted_SCL_Profile'].drop_duplicates()
    return len(unique_scls) > 1

final_candidates = merged_df.groupby('Gene').filter(has_adjusted_scl_variance)

# Format and Export
final_output = final_candidates[['Gene', 'Transcript', 'RBP_Profile', 'Adjusted_SCL_Profile']].copy()
final_output.sort_values(by=['Gene', 'Transcript'], inplace=True)
final_output.to_csv(output_file, index=False)

print(f"Total transcripts inputted from variance matrix: {len(matrix_df)}")
print(f"Transcripts successfully mapped to prediction file: {len(merged_df)}")
print(f"Transcripts retained after dropping uniform macro-SCL genes: {len(final_output)}")
print(f"Total high-confidence candidate genes: {final_output['Gene'].nunique()}")
print(f"\nFinal adjusted dataset saved to: {output_file}")

Total transcripts inputted from variance matrix: 440
Transcripts successfully mapped to prediction file: 440
Transcripts retained after dropping uniform macro-SCL genes: 338
Total high-confidence candidate genes: 90

Final adjusted dataset saved to: Data/FIMO_New/Adjusted_Functional_Zipcode_Candidates.csv
